# k Retrieval Dependence

This notebook compares semantic retrieval results for `qwen7b` across `k=1,2,3`, using the same experiment labels as the other retrieval notebooks.

In [14]:
import os
import sys
from pathlib import Path

RETRIEVAL_DIR = Path.cwd().resolve()
REPO_ROOT = RETRIEVAL_DIR.parent
LOCAL_RESULTS_ROOT = RETRIEVAL_DIR / 'results'
LEGACY_RESULTS_ROOT = REPO_ROOT / 'results'
LOCAL_RESULTS_RECALL_SEM = str(LOCAL_RESULTS_ROOT / 'recall' / 'sem') + '/'
LEGACY_RESULTS_RECALL_SEM = str(LEGACY_RESULTS_ROOT / 'recall' / 'sem') + '/'
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils import makefolder
import numpy as np
from IPython.display import Markdown, display

K_VALUES = [1, 2, 3]
ROW_LABELS = [
    'baseline (no ablation)',
    'semantic ablation',
    'syntactic ablation',
    'semantic ablation (random)',
    'syntactic ablation (random)',
]
METRIC_KEYS = [
    'recalls_0',
    'recalls_sem',
    'recalls_syn',
    'recalls_sem_perm',
    'recalls_syn_perm',
]
DISPLAY_NAMES = {
    'qwen7b': 'Qwen2-7b',
    'deepseek': 'DeepSeek',
    'gemma12b': 'Gemma-12b',
}


def get_recall_path(model_name, k_recall, avg_tokens=1, min_token_length=3, global_center_flag=1):
    local_folder = makefolder(
        base=LOCAL_RESULTS_RECALL_SEM,
        create_folder=False,
        model_name=model_name,
        avg_tokens=avg_tokens,
        min_token_length=min_token_length,
        global_center_flag=global_center_flag,
        k=k_recall,
    )
    local_path = os.path.join(local_folder, f'recall_k{k_recall}.npz')
    if os.path.exists(local_path):
        return local_path

    legacy_folder = makefolder(
        base=LEGACY_RESULTS_RECALL_SEM,
        create_folder=False,
        model_name=model_name,
        avg_tokens=avg_tokens,
        min_token_length=min_token_length,
        global_center_flag=global_center_flag,
        k=k_recall,
    )
    return os.path.join(legacy_folder, f'recall_k{k_recall}.npz')


def load_best_recalls(model_name, avg_tokens=1, min_token_length=3, global_center_flag=1, k_values=None):
    if k_values is None:
        k_values = K_VALUES

    best_recalls_by_k = {}
    for k_recall in k_values:
        results_path = get_recall_path(
            model_name=model_name,
            k_recall=k_recall,
            avg_tokens=avg_tokens,
            min_token_length=min_token_length,
            global_center_flag=global_center_flag,
        )
        data = np.load(results_path)
        best_recalls_by_k[k_recall] = {
            label: float(np.max(data[key]))
            for label, key in zip(ROW_LABELS, METRIC_KEYS)
        }
    return best_recalls_by_k


def format_cell(value, min_value, max_value):
    text = f'{value:.4f}'
    if np.isclose(value, min_value):
        text = f'**{text}**'
    if np.isclose(value, max_value):
        text = f'<u>{text}</u>'
    return text


def render_model_table(model_name, avg_tokens=1, min_token_length=3, global_center_flag=1, k_values=None):
    if k_values is None:
        k_values = K_VALUES

    best_recalls_by_k = load_best_recalls(
        model_name=model_name,
        avg_tokens=avg_tokens,
        min_token_length=min_token_length,
        global_center_flag=global_center_flag,
        k_values=k_values,
    )

    column_min = {
        k: min(best_recalls_by_k[k][label] for label in ROW_LABELS)
        for k in k_values
    }
    column_max = {
        k: max(best_recalls_by_k[k][label] for label in ROW_LABELS)
        for k in k_values
    }

    display_name = DISPLAY_NAMES.get(model_name, model_name)
    lines = [f'### {display_name}', '']
    lines.append('| Paraphrase-recall@k | ' + ' | '.join(f'k={k}' for k in k_values) + ' |')
    lines.append('| ' + ' | '.join(['---'] * (len(k_values) + 1)) + ' |')

    for label in ROW_LABELS:
        values = [
            format_cell(best_recalls_by_k[k][label], column_min[k], column_max[k])
            for k in k_values
        ]
        lines.append('| ' + ' | '.join([label] + values) + ' |')

    markdown_table = '\n'.join(lines)
    display(Markdown(markdown_table))
    return markdown_table


In [15]:
render_model_table('qwen7b')


### Qwen2-7b

| Paraphrase-recall@k | k=1 | k=2 | k=3 |
| --- | --- | --- | --- |
| baseline (no ablation) | 0.7312 | 0.8600 | 0.9062 |
| semantic ablation | **0.5300** | **0.6456** | **0.7081** |
| syntactic ablation | <u>0.7587</u> | <u>0.8769</u> | <u>0.9187</u> |
| semantic ablation (random) | 0.7188 | 0.8275 | 0.8806 |
| syntactic ablation (random) | 0.7275 | 0.8531 | 0.9038 |

'### Qwen2-7b\n\n| Paraphrase-recall@k | k=1 | k=2 | k=3 |\n| --- | --- | --- | --- |\n| baseline (no ablation) | 0.7312 | 0.8600 | 0.9062 |\n| semantic ablation | **0.5300** | **0.6456** | **0.7081** |\n| syntactic ablation | <u>0.7587</u> | <u>0.8769</u> | <u>0.9187</u> |\n| semantic ablation (random) | 0.7188 | 0.8275 | 0.8806 |\n| syntactic ablation (random) | 0.7275 | 0.8531 | 0.9038 |'

In [16]:
render_model_table('deepseek')


### DeepSeek

| Paraphrase-recall@k | k=1 | k=2 | k=3 |
| --- | --- | --- | --- |
| baseline (no ablation) | 0.6994 | 0.8156 | 0.8531 |
| semantic ablation | **0.5375** | **0.6394** | **0.6900** |
| syntactic ablation | <u>0.7531</u> | <u>0.8706</u> | <u>0.9044</u> |
| semantic ablation (random) | 0.6706 | 0.7756 | 0.8287 |
| syntactic ablation (random) | 0.6950 | 0.8050 | 0.8475 |

'### DeepSeek\n\n| Paraphrase-recall@k | k=1 | k=2 | k=3 |\n| --- | --- | --- | --- |\n| baseline (no ablation) | 0.6994 | 0.8156 | 0.8531 |\n| semantic ablation | **0.5375** | **0.6394** | **0.6900** |\n| syntactic ablation | <u>0.7531</u> | <u>0.8706</u> | <u>0.9044</u> |\n| semantic ablation (random) | 0.6706 | 0.7756 | 0.8287 |\n| syntactic ablation (random) | 0.6950 | 0.8050 | 0.8475 |'

In [17]:
render_model_table('gemma12b')


### Gemma-12b

| Paraphrase-recall@k | k=1 | k=2 | k=3 |
| --- | --- | --- | --- |
| baseline (no ablation) | 0.4244 | 0.5012 | 0.5537 |
| semantic ablation | **0.3256** | **0.3837** | **0.4206** |
| syntactic ablation | <u>0.4719</u> | <u>0.5869</u> | <u>0.6431</u> |
| semantic ablation (random) | 0.3981 | 0.4894 | 0.5344 |
| syntactic ablation (random) | 0.4181 | 0.4944 | 0.5381 |

'### Gemma-12b\n\n| Paraphrase-recall@k | k=1 | k=2 | k=3 |\n| --- | --- | --- | --- |\n| baseline (no ablation) | 0.4244 | 0.5012 | 0.5537 |\n| semantic ablation | **0.3256** | **0.3837** | **0.4206** |\n| syntactic ablation | <u>0.4719</u> | <u>0.5869</u> | <u>0.6431</u> |\n| semantic ablation (random) | 0.3981 | 0.4894 | 0.5344 |\n| syntactic ablation (random) | 0.4181 | 0.4944 | 0.5381 |'